In [1]:
# Import agent configuration and core libraries
import sys
import os

# Add the project root to Python path so we can import from agent
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from agent.core.config import settings
from langchain_ollama import ChatOllama
import httpx
import json

In [3]:
# Test connection to services
print("=== Service Configuration ===")
print(f"World API: {settings.WORLD_API_URL}")
print(f"Neo4j: {settings.NEO4J_URI}")
print(f"Ollama: {settings.OLLAMA_BASE_URL}")
print(f"Environment: {settings.ENVIRONMENT}")

try:
    response = httpx.get(f"{settings.WORLD_API_URL}/docs")
    print(f"\n✅ World API: Connected (Status: {response.status_code})")
except Exception as e:
    print(f"\n❌ World API: Failed to connect - {e}")
# Test Ollama
try:
    llm = ChatOllama(base_url=settings.OLLAMA_BASE_URL, model="llama3.1:8b")
    print(settings.OLLAMA_BASE_URL)
    print("\n✅ Ollama: Connected")
except Exception as e:
    print(f"\n❌ Ollama: Failed to connect - {e}")


=== Service Configuration ===
World API: http://world:8000
Neo4j: bolt://neo4j:7687
Ollama: http://host.docker.internal:11434
Environment: development

✅ World API: Connected (Status: 200)
http://host.docker.internal:11434

✅ Ollama: Connected


In [ ]:
#Parse feed from json into a LLM friendly format using pydantic

from datetime import datetime
from typing import List
from pydantic import BaseModel, Field

class Author(BaseModel):
    name: str
    persona: str
    id: int
    created_at: datetime

class Comment(BaseModel):
    text: str
    author_id: int
    id: int
    created_at: datetime

class Post(BaseModel):
    text: str
    agent_id: int
    id: int
    created_at: datetime
    author: Author
    comments: List[Comment] = Field(default_factory=list)

class Feed(BaseModel):
        posts: List[Post]

        def to_formatted_text(self) -> str:
            """
            Generates a human-readable, report-style string representation of the feed.
            """
            formatted_parts = []
            
            for post in self.posts:
                post_time = post.created_at.strftime("%Y-%m-%d %H:%M:%S")

                formatted_parts.append(f"Post ID: {post.id}")
                formatted_parts.append(f"Author: {post.author.name} (Persona: {post.author.persona})")
                formatted_parts.append(f"Posted On: {post_time}")
                formatted_parts.append(f"Text: {post.text}")
                
                comment_text = "None" if not post.comments else f"{len(post.comments)} comment(s)"
                formatted_parts.append(f"Comments: {comment_text}")
                
                if post != self.posts[-1]:
                    formatted_parts.append("\n---\n")
            return "\n".join(formatted_parts)

In [14]:
# Test the world_tools functions
from agent.tools.world_tools import get_agent, get_agents

# First, let's see what agents exist
print("=== Available Agents ===")
agents = get_agents()
print(f"Total agents: {len(agents)}")
for agent in agents:
    print(f"Agent {agent['id']}: {agent['name']} - {agent['persona']}")

print("\n=== Testing get_agent function ===")
if agents:
    # Try to get the first available agent
    first_agent_id = agents[0]['id']
    print(f"Trying to get agent with ID {first_agent_id}")
    agent = get_agent(first_agent_id)
    if agent:
        print(f"✅ Successfully retrieved agent: {agent['name']}")
    else:
        print(f"❌ Failed to retrieve agent {first_agent_id}")
else:
    print("No agents found in the world")


ERROR:agent.tools.world_tools:Request failed for /api/agents/1: 404 Client Error: Not Found for url: http://world:8000/api/agents/1


=== Available Agents ===
Total agents: 1
Agent 1: adam - The first agent.

=== Testing get_agent function ===
Trying to get agent with ID 1
❌ Failed to retrieve agent 1


In [17]:
# Debug the API endpoints directly
import httpx

# Test the endpoints directly to see what's happening
headers = {
    'X-API-Key': settings.API_KEY,
    'Content-Type': 'application/json'
}

print("=== Direct API Testing ===")

# Test 1: GET /api/agents/ (this works)
print("1. Testing GET /api/agents/")
try:
    response = httpx.get(f"{settings.WORLD_API_URL}/api/agents/", headers=headers)
    print(f"   Status: {response.status_code}")
    agents = response.json()
    print(f"   Agents found: {len(agents)}")
    if agents:
        print(f"   First agent: ID={agents[0]['id']}, name='{agents[0]['name']}'")
except Exception as e:
    print(f"   Error: {e}")

# Test 2: GET /api/agents/1 (this fails)
print("\n2. Testing GET /api/agents/1")
try:
    response = httpx.get(f"{settings.WORLD_API_URL}/api/agents/1", headers=headers)
    print(f"   Status: {response.status_code}")
    if response.status_code == 200:
        agent = response.json()
        print(f"   Agent: {agent}")
    else:
        print(f"   Error response: {response.text}")
except Exception as e:
    print(f"   Error: {e}")

# Test 3: Check if the endpoint exists at all
print("\n3. Testing available endpoints")
try:
    response = httpx.get(f"{settings.WORLD_API_URL}/docs")
    print(f"   API docs accessible: {response.status_code == 200}")
except Exception as e:
    print(f"   Error accessing docs: {e}")

# Test 4: Check what happens with different agent IDs
print("\n4. Testing different approaches")
print("   Available agents from /api/agents/:")
try:
    response = httpx.get(f"{settings.WORLD_API_URL}/api/agents/", headers=headers)
    agents = response.json()
    for agent in agents:
        agent_id = agent['id']
        print(f"   Trying /api/agents/{agent_id}")
        try:
            single_response = httpx.get(f"{settings.WORLD_API_URL}/api/agents/{agent_id}", headers=headers)
            print(f"     Status: {single_response.status_code}")
            if single_response.status_code != 200:
                print(f"     Error: {single_response.text}")
        except Exception as e2:
            print(f"     Error: {e2}")
except Exception as e:
    print(f"   Error: {e}")


INFO:httpx:HTTP Request: GET http://world:8000/api/agents/ "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET http://world:8000/api/agents/1 "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET http://world:8000/docs "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://world:8000/api/agents/ "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://world:8000/api/agents/1 "HTTP/1.1 404 Not Found"


=== Direct API Testing ===
1. Testing GET /api/agents/
   Status: 200
   Agents found: 1
   First agent: ID=1, name='adam'

2. Testing GET /api/agents/1
   Status: 404
   Error response: {"detail":"Not Found"}

3. Testing available endpoints
   API docs accessible: True

4. Testing different approaches
   Available agents from /api/agents/:
   Trying /api/agents/1
     Status: 404
     Error: {"detail":"Not Found"}


In [19]:
from agent.tools.world_tools import get_agent

get_agent(1)

{'name': 'adam',
 'persona': 'The first agent.',
 'id': 1,
 'created_at': '2025-08-20T17:50:43.021778'}